In [13]:
import sqlite3

DB_FILE = "aling_puring.db"

def init_db():
    """Gagawa ng koneksyon at mga tables kung wala pa ito sa computer."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    # Requirement #1: Table para sa Multiple Branches
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS branches (
        id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        location TEXT NOT NULL
    )""")
    
    # Requirement #1 & #3: Table para sa Inventories ng bawat Branch
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS inventory (
        branch_id TEXT,
        product_name TEXT,
        price REAL NOT NULL,
        stock INTEGER NOT NULL,
        restock_level INTEGER NOT NULL,
        PRIMARY KEY (branch_id, product_name),
        FOREIGN KEY (branch_id) REFERENCES branches(id) ON DELETE CASCADE
    )""")
    
    # Requirement #2: Table para sa Sales Transactions
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS transactions (
        txn_id INTEGER PRIMARY KEY AUTOINCREMENT,
        branch_id TEXT,
        total_amount REAL NOT NULL,
        FOREIGN KEY (branch_id) REFERENCES branches(id)
    )""")
    
    conn.commit()
    conn.close()
    print("Database at mga tables: OK at Handa na!")

# Patakbuhin ang initialization
init_db()

Database at mga tables: OK at Handa na!


In [15]:
def add_branch(b_id, name, location):
    """Magdagdag ng bagong branch (Requirement #1)"""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO branches (id, name, location) VALUES (?, ?, ?)", (b_id, name, location))
        conn.commit()
        print(f"Branch '{name}' ay matagumpay na na-save!")
    except sqlite3.IntegrityError:
        print(f"Ang Branch ID o pangalang '{name}' ay umiiral na.")
    finally:
        conn.close()

def add_product(branch_id, p_name, price, stock, restock):
    """Magdagdag ng produkto sa isang partikular na branch (Separate Inventories)"""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("""
        INSERT OR REPLACE INTO inventory (branch_id, product_name, price, stock, restock_level) 
        VALUES (?, ?, ?, ?, ?)""", (branch_id, p_name, price, stock, restock))
    conn.commit()
    conn.close()
    print(f"Product '{p_name}' ay nai-save sa inventory ng {branch_id}.")

def record_sale(branch_id, p_name, qty_to_sell):
    """Mag-record ng benta at awtomatikong magbawas ng stock level (Requirement #2 & #3)"""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    # Tignan muna kung may sapat na stock
    cursor.execute("SELECT price, stock FROM inventory WHERE branch_id = ? AND product_name = ?", (branch_id, p_name))
    row = cursor.fetchone()
    
    if not row:
        print("Paumanhin, hindi mahanap ang produkto sa branch na ito.")
        conn.close()
        return
        
    price, current_stock = row
    if qty_to_sell > current_stock:
        print(f"Kulang ang stock! Mayroon na lamang {current_stock} piraso ng {p_name}.")
        conn.close()
        return
        
    # 1. AUTOMATICALLY UPDATE STOCK LEVELS (Requirement #3)
    new_stock = current_stock - qty_to_sell
    cursor.execute("UPDATE inventory SET stock = ? WHERE branch_id = ? AND product_name = ?", (new_stock, branch_id, p_name))
    
    # 2. RECORD SALES TRANSACTION (Requirement #2)
    total = price * qty_to_sell
    cursor.execute("INSERT INTO transactions (branch_id, total_amount) VALUES (?, ?)", (branch_id, total))
    
    conn.commit()
    conn.close()
    
    print("\n========================================")
    print("           SALES TRANSACTION SUCCESS")
    print(f" Product: {p_name} | Qty Sold: {qty_to_sell}")
    print(f" Total Paid: P{total:.2f}")
    print(f" Remaining Stock: {new_stock}")
    print("========================================")

def view_branch_report(branch_id):
    """Mag-generate ng sales report kada branch (Requirement #4)"""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    cursor.execute("SELECT SUM(total_amount), COUNT(txn_id) FROM transactions WHERE branch_id = ?", (branch_id,))
    total_sales, txn_count = cursor.fetchone()
    
    total_sales = total_sales if total_sales else 0.0
    print(f"\n=== SALES REPORT FOR BRANCH: {branch_id.upper()} ===")
    print(f" Total Transactions : {txn_count}")
    print(f" Total Gross Revenue: P{total_sales:.2f}")
    print("========================================")
    conn.close()

def view_overall_report():
    """Mag-generate ng kabuuang ulat sa buong negosyo (Requirement #4)"""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    cursor.execute("SELECT SUM(total_amount), COUNT(txn_id) FROM transactions")
    grand_total, total_txns = cursor.fetchone()
    
    grand_total = grand_total if grand_total else 0.0
    print("\n=== OVERALL BUSINESS REPORT ===")
    print(f" Total Orders Processed (All Branches): {total_txns}")
    print(f" Grand Total Revenue (All Branches)  : P{grand_total:.2f}")
    print("========================================")
    conn.close()

In [16]:
# --- STEP 1: Subukang magdagdag ng Multiple Branches ---
print("--- Pag-add ng mga Branches ---")
add_branch("main_branch", "Main Branch", "Antipolo")
add_branch("sub_branch", "Sub Branch", "Dela Paz")

# --- STEP 2: Maglagay ng paninda (Mapapansin na magkabukod ang kanilang inventory) ---
print("\n--- Pag-add ng mga Produkto sa Inventory ---")
add_product("main_branch", "Coke Original", 20.0, 50, 5)
add_product("main_branch", "Pancit Canton", 15.0, 4, 5)
add_product("sub_branch", "Sardines", 22.50, 30, 5)

# --- STEP 3: Magbenta (Dito makikita ang awtomatikong pagbabawas ng stock) ---
print("\n--- Pag-execute ng Benta (Sales) ---")
record_sale("main_branch", "Coke Original", 5)
record_sale("main_branch", "Pancit Canton", 2)

# --- STEP 4: Pag-generate ng mga Ulat (Reports) ---
print("\n--- Pag-view ng mga Ulat ---")
view_branch_report("main_branch")
view_overall_report()

--- Pag-add ng mga Branches ---
Ang Branch ID o pangalang 'Main Branch' ay umiiral na.
Ang Branch ID o pangalang 'Sub Branch' ay umiiral na.

--- Pag-add ng mga Produkto sa Inventory ---
Product 'Coke Original' ay nai-save sa inventory ng main_branch.
Product 'Pancit Canton' ay nai-save sa inventory ng main_branch.
Product 'Sardines' ay nai-save sa inventory ng sub_branch.

--- Pag-execute ng Benta (Sales) ---

           SALES TRANSACTION SUCCESS
 Product: Coke Original | Qty Sold: 5
 Total Paid: P100.00
 Remaining Stock: 45

           SALES TRANSACTION SUCCESS
 Product: Pancit Canton | Qty Sold: 2
 Total Paid: P30.00
 Remaining Stock: 2

--- Pag-view ng mga Ulat ---

=== SALES REPORT FOR BRANCH: MAIN_BRANCH ===
 Total Transactions : 4
 Total Gross Revenue: P330.00

=== OVERALL BUSINESS REPORT ===
 Total Orders Processed (All Branches): 4
 Grand Total Revenue (All Branches)  : P330.00
